<a href="https://colab.research.google.com/github/BF667/UVRC/blob/main/UVRC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 UVRC — Ultimate Vocal Remover Colab

State-of-the-art audio source separation powered by deep learning.

## Quick Start
1. **Run Cell 1** to install UVRC and set up directories.
2. **Upload your audio** to the `input/` folder (or connect Google Drive).
3. **Run Cell 2**, configure the model and settings, then wait for separation.
4. **Run Cell 3** to play back the results.

## Supported Models
80+ pre-trained models including: vocals, instrumental, drums, de-reverb, denoise, karaoke, crowd removal, guitar, and more.

In [ ]:
#@markdown # 1️⃣ Install UVRC

import os, sys

%cd /content

# Install the package
!pip install -q git+https://github.com/BF667/UVRC.git

# Create required directories
os.makedirs('ckpts', exist_ok=True)
os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️  No GPU detected — separation will be very slow on CPU.')
    print('   Go to Runtime → Change runtime type → T4 GPU')

print('\n✅ Installation complete!')

In [ ]:
#@markdown # 2️⃣ Separate Audio
#@markdown <hr/>
#@markdown ### 📥 Model Configuration
#@markdown Paste config and checkpoint URLs, or leave blank to use the preset model dropdown below.
config_url = '' #@param {type:"string"}
ckpt_url = '' #@param {type:"string"}
model_type = 'mel_band_roformer' #@param ['mdx23c','bs_roformer', 'mel_band_roformer', 'bandit', 'bandit_v2', 'scnet', 'htdemucs', 'segm_models', 'torchseg']

#@markdown <hr/>
#@markdown ### 🎛️ Separation Settings
input_file = '/content/input/audio.mp3' #@param {type:"string"}
output_folder = '/content/output' #@param {type:"string"}
extract_instrumental = True #@param {type:"boolean"}
export_format = 'flac PCM_16' #@param ['wav FLOAT', 'flac PCM_16', 'flac PCM_24']
use_tta = False #@param {type:"boolean"}

#@markdown <hr/>
#@markdown ### ⚙️ Roformer Custom Config
#@markdown *Only applies to `bs_roformer` and `mel_band_roformer` models.*
overlap = 4 #@param {type:"slider", min:2, max:40, step:1}
chunk_size = "485100" #@param [352800, 485100] {allow-input: true}

# ── Resolve format ──────────────────────────────────────────────
if export_format.startswith('flac'):
    flac_file = True
    pcm_type = export_format.split(' ')[1]
else:
    flac_file = False
    pcm_type = None

# ── Validate input file ─────────────────────────────────────────
import os as _os
if not _os.path.isfile(input_file):
    print(f'⚠️  Input file not found: {input_file}')
    print('   Please upload your audio file to the input/ folder first.')
    print('   Or update the input_file path above.')

# ── Download model files ────────────────────────────────────────
if config_url and ckpt_url:
    # Custom URL mode
    from UVRC.multi import download_file, conf_edit

    config_filename = _os.path.basename(config_url)
    ckpt_filename = _os.path.basename(ckpt_url)
    print(f'Downloading model files…')
    print(f'  Config: {config_filename}')
    print(f'  Checkpoint: {ckpt_filename}')

    config_path = f'ckpts/{config_filename}'
    start_check_point = f'ckpts/{ckpt_filename}'
    download_file(config_url)
    download_file(ckpt_url)

    # Edit config for roformer models
    if 'roformer' in model_type:
        conf_edit(config_path, int(chunk_size), overlap)
elif not config_url and not ckpt_url:
    # No URLs provided — user must supply config_path and start_check_point via CLI
    print('⚠️  No model URLs provided. Please fill in config_url and ckpt_url above.')
    print('   You can find model URLs at: https://github.com/ZFTurbo/Music-Source-Separation-Training')

# ── Run separation ──────────────────────────────────────────────
if config_url and ckpt_url and _os.path.isfile(input_file):
    cmd_parts = [
        'uvr-cli',
        f'--model_type {model_type}',
        f"--config_path '{config_path}'",
        f"--start_check_point '{start_check_point}'",
        f"--input_file '{input_file}'",
        f"--store_dir '{output_folder}'",
    ]
    if extract_instrumental:
        cmd_parts.append('--extract_instrumental')
    if flac_file:
        cmd_parts.append('--flac_file')
    if use_tta:
        cmd_parts.append('--use_tta')
    if pcm_type:
        cmd_parts.append(f'--pcm_type {pcm_type}')

    cmd = ' \\\n    '.join(cmd_parts)
    print(f'\n🎵 Running separation…\n')
    get_ipython().system(cmd)
    print(f'\n✅ Separation complete! Check output folder: {output_folder}')
else:
    print('\n❌ Cannot run separation — missing model URLs or input file.')

In [ ]:
#@markdown # 3️⃣ Play Results

import os, glob
from IPython.display import Audio, display

#@markdown Leave blank to auto-detect the most recent output file.
audio_path = '' #@param {type:"string"}

if not audio_path:
    # Auto-detect latest output file
    output_files = sorted(
        glob.glob('/content/output/*.wav') + glob.glob('/content/output/*.flac'),
        key=os.path.getmtime,
        reverse=True
    )
    if output_files:
        audio_path = output_files[0]
        print(f'Playing: {os.path.basename(audio_path)}')
        if len(output_files) > 1:
            print(f'  ({len(output_files)-1} more files in output/)')
    else:
        print('No output files found in /content/output/')

if audio_path and os.path.isfile(audio_path):
    display(Audio(audio_path))
elif audio_path:
    print(f'File not found: {audio_path}')

## 📝 Notes

### Swapped labels
**INST-Mel-Roformer v1 / 1e / 2** has switched output file names — files labelled as *vocals* are actually instrumentals. If you uncheck *extract_instrumental* for the v1e model, only one stem called "other" will be rendered, and it will be the instrumental.

### TTA (Test-Time Augmentation)
Enabling **TTA** triples the runtime but can slightly improve quality. It performs 3 passes: original audio, inverted stereo (L↔R), and phase-inverted, then averages the results.

### Overlap
Higher overlap = longer separation time. **4** is a balanced value; **2** is fast with minimal audible difference. There's normally no point going above **8**.

### Troubleshooting
- **"Total files found: 0"** — Make sure `input_file` points to an actual audio file, not a folder. The Colab is case-sensitive (use `input` not `Input`).
- **GPU not detected** — Go to *Runtime → Change runtime type → T4 GPU*.
- **Out of memory** — Try reducing `chunk_size` to `352800` or lowering `overlap`.

In [ ]:
#@markdown # 🔄 Force Google Drive Remount
#@markdown Run this if your Drive folder appears empty in the file manager.
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
print('✅ Drive remounted!')